In [15]:
import os
print(os.getcwd())

/workspace/notebooks


In [26]:
import zipfile
import yaml

zip_path = "/course-data/raw/source=google_qec/google-surface-code-curated.zip"

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()

experiment_folders = sorted({n.split("/")[0] for n in names if "/" in n})
print(experiment_folders)

['surface_code_bX_d3_r25_center_3_5', 'surface_code_bX_d3_r25_center_5_3', 'surface_code_bX_d3_r25_center_5_7', 'surface_code_bX_d3_r25_center_7_5', 'surface_code_bX_d5_r25_center_5_5']


In [29]:
with zipfile.ZipFile(zip_path) as z:
    with z.open("surface_code_bX_d3_r25_center_3_5/properties.yml") as f:
        properties = yaml.safe_load(f)
print(properties)

{'type': 'surface_code_memory_experiment', 'basis': 'X', 'rounds': 25, 'distance': 3, 'data_qubits': 9, 'measure_qubits': 8, 'shots': 50000, 'center_data_qubit_row': 3, 'center_data_qubit_col': 5, 'circuit_measurements': 209, 'circuit_sweep_bits': 9, 'circuit_detectors': 200, 'circuit_observables': 1, 'circuit_qubits': 17}


In [25]:
-import importlib
import Utility
importlib.reload(Utility)
from Utility import Utility as Helper

zip_path = "/course-data/raw/source=google_qec/google-surface-code-curated.zip"


folder = Helper.list_zip_folders(zip_path)[0]
properties = Helper.read_zip_member_yaml(zip_path, f"{folder}/properties.yml")
measurements = Helper.read_b8_records(zip_path, f"{folder}/measurements.b8", properties["circuit_measurements"])
actual_flips = Helper.read_01_records(zip_path, f"{folder}/obs_flips_actual.01")
Helper.check_equal_lengths(measurements=measurements, actual_flips=actual_flips)

OK: all 2 lists have length 50000


True

In [30]:
import importlib
import Utility
importlib.reload(Utility)
from Utility import Utility as Helper

import pandas as pd

zip_path = "/course-data/raw/source=google_qec/google-surface-code-curated.zip"

folders = Helper.list_zip_folders(zip_path)
print(folders)

# start with just one folder to sanity-check before looping over all 5
rows = Helper.build_shot_rows(zip_path, folders[0])
shot_df = pd.DataFrame(rows)

print(shot_df.shape)
shot_df.head()

['surface_code_bX_d3_r25_center_3_5', 'surface_code_bX_d3_r25_center_5_3', 'surface_code_bX_d3_r25_center_5_7', 'surface_code_bX_d3_r25_center_7_5', 'surface_code_bX_d5_r25_center_5_5']
OK: all 8 lists have length 50000
(50000, 12)


,source_record_id,experiment_id,shot_index,measurement_bits,sweep_bits,detector_bits,detector_event_count,actual_observable_flip,belief_matching_prediction,correlated_matching_prediction,pymatching_prediction,tensor_network_contraction_prediction
0,None,surface_code_bX_d3_r25_center_3_5,0,b'\x85 \xad\x905\x90_\xb8\x1d\xb8\x1d\xb8]\xf8...,b'\xd0\x00',b'\x00\x80\x82\t\x00\xa0&\x04\x00\x00\x00\x00\...,27,True,True,True,True,True
1,None,surface_code_bX_d3_r25_center_3_5,1,b'\xac\x02\xaf<\x9dp\xcd\xb8\x9dx\xddy\xbcY\xf...,b'\xd0\x00',b'\xb3\x80`C\x80\x84\x01\r\x08\x04\x10\x00\x06...,36,True,True,False,False,True
2,None,surface_code_bX_d3_r25_center_3_5,2,b'\xd3\xf4S\xb6S\xb6\x11\xa6\x13\xe6\xc1l\xfb\...,b'\xd0\x00',b'$(\x00\x04\x04$ \x01\x01%\x88 \xa3\x04\x01\x...,32,True,True,True,False,True
3,None,surface_code_bX_d3_r25_center_3_5,3,b'\x83$\x83%\xb27\x927\x927\x927\xf1T\xe5P\xf5...,b'\xd0\x00',b' 0 \x03\x02\x00\x00\x00\x00\x000\x06@\x01\x...,25,False,False,True,True,False
4,None,surface_code_bX_d3_r25_center_3_5,4,b'\xe1D\xe2\x15\xa3\x11\xc4\x83\x06\xa2\x1b\xb...,b'\xd0\x00',"b""\x000 5q\x01'\x0e\x12\xc0\xc1\x80\x00\x08iHA...",52,True,False,False,False,True


In [31]:
all_rows = []
for folder in folders:
    all_rows.extend(Helper.build_shot_rows(zip_path, folder))

shot_df = pd.DataFrame(all_rows)
print(shot_df.shape)

OK: all 8 lists have length 50000
OK: all 8 lists have length 50000
OK: all 8 lists have length 50000
OK: all 8 lists have length 50000
OK: all 8 lists have length 50000
(250000, 12)


In [33]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 13.6 MB/s eta 0:00:00a 0:00:01


In [37]:
import importlib
import boto3
import pandas as pd
import Utility

importlib.reload(Utility)
from Utility import Utility as Helper

# 1. Connect to S3
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="quantum",
    aws_secret_access_key="quantum-course-only",
    region_name="us-east-1",
)
obj = s3.get_object(
    Bucket="quantum-lake",
    Key="bronze/source=google_qec/google-surface-code-curated.zip",
)
raw_bytes = obj["Body"].read()

# 2. Inspect folders
folders = Helper.list_zip_folders(raw_bytes)
print("Found folders:", folders)

# 3. Build shot rows (single folder test or loop)
all_shot_rows = []
for folder in folders:
    rows = Helper.build_shot_rows(raw_bytes, folder)
    all_shot_rows.extend(rows)

shot_df = pd.DataFrame(all_shot_rows)
shot_df.head()

,source_record_id,experiment_id,shot_index,measurement_bits,sweep_bits,detector_bits,detector_event_count,actual_observable_flip,belief_matching_prediction,correlated_matching_prediction,pymatching_prediction,tensor_network_contraction_prediction
0,google_shot_81ebcd9698dc,google_surface_code_bX_d3_r25_center_3_5,0,b'\x85 \xad\x905\x90_\xb8\x1d\xb8\x1d\xb8]\xf8...,b'\xd0\x00',b'\x00\x80\x82\t\x00\xa0&\x04\x00\x00\x00\x00\...,27,True,True,True,True,True
1,google_shot_4959c10c62f4,google_surface_code_bX_d3_r25_center_3_5,1,b'\xac\x02\xaf<\x9dp\xcd\xb8\x9dx\xddy\xbcY\xf...,b'\xd0\x00',b'\xb3\x80`C\x80\x84\x01\r\x08\x04\x10\x00\x06...,36,True,True,False,False,True
2,google_shot_16102497aa63,google_surface_code_bX_d3_r25_center_3_5,2,b'\xd3\xf4S\xb6S\xb6\x11\xa6\x13\xe6\xc1l\xfb\...,b'\xd0\x00',b'$(\x00\x04\x04$ \x01\x01%\x88 \xa3\x04\x01\x...,32,True,True,True,False,True
3,google_shot_c43152e2503c,google_surface_code_bX_d3_r25_center_3_5,3,b'\x83$\x83%\xb27\x927\x927\x927\xf1T\xe5P\xf5...,b'\xd0\x00',b' 0 \x03\x02\x00\x00\x00\x00\x000\x06@\x01\x...,25,False,False,True,True,False
4,google_shot_173225b589cd,google_surface_code_bX_d3_r25_center_3_5,4,b'\xe1D\xe2\x15\xa3\x11\xc4\x83\x06\xa2\x1b\xb...,b'\xd0\x00',"b""\x000 5q\x01'\x0e\x12\xc0\xc1\x80\x00\x08iHA...",52,True,False,False,False,True


In [38]:
shot_df.columns

Index(['source_record_id', 'experiment_id', 'shot_index', 'measurement_bits',
       'sweep_bits', 'detector_bits', 'detector_event_count',
       'actual_observable_flip', 'belief_matching_prediction',
       'correlated_matching_prediction', 'pymatching_prediction',
       'tensor_network_contraction_prediction'],
      dtype='object')

In [2]:
import pandas as pd

In [6]:
df = pd.read_parquet('./../results/part1/data_issues.parquet')
print(df.shape)
df.head(500)

(80, 8)


,issue_id,run_id,source_record_id,rule_id,severity,observed_value,action,reason
0,issue_9890e823e7c0,8f0f9172-f557-4931-814f-49a5ec137565,qasmbench:small/error_correctiond3_n5/error_co...,qasm_circuit_known_gate,warning,sx,accepted,gate is neither defined in the circuit nor dec...
1,issue_9890e823e7c0,8f0f9172-f557-4931-814f-49a5ec137565,qasmbench:small/error_correctiond3_n5/error_co...,qasm_circuit_known_gate,warning,sx,accepted,gate is neither defined in the circuit nor dec...
2,issue_9890e823e7c0,8f0f9172-f557-4931-814f-49a5ec137565,qasmbench:small/error_correctiond3_n5/error_co...,qasm_circuit_known_gate,warning,sx,accepted,gate is neither defined in the circuit nor dec...
3,issue_9890e823e7c0,8f0f9172-f557-4931-814f-49a5ec137565,qasmbench:small/error_correctiond3_n5/error_co...,qasm_circuit_known_gate,warning,sx,accepted,gate is neither defined in the circuit nor dec...
4,issue_9890e823e7c0,8f0f9172-f557-4931-814f-49a5ec137565,qasmbench:small/error_correctiond3_n5/error_co...,qasm_circuit_known_gate,warning,sx,accepted,gate is neither defined in the circuit nor dec...
...,...,...,...,...,...,...,...,...
75,issue_a5a9e7fa0f42,8f0f9172-f557-4931-814f-49a5ec137565,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm,qasm_circuit_known_gate,warning,if,accepted,gate is neither defined in the circuit nor dec...
76,issue_a5a9e7fa0f42,8f0f9172-f557-4931-814f-49a5ec137565,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm,qasm_circuit_known_gate,warning,if,accepted,gate is neither defined in the circuit nor dec...
77,issue_bf1367a33128,8f0f9172-f557-4931-814f-49a5ec137565,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled...,qasm_circuit_known_gate,warning,if,accepted,gate is neither defined in the circuit nor dec...
78,issue_bf1367a33128,8f0f9172-f557-4931-814f-49a5ec137565,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled...,qasm_circuit_known_gate,warning,if,accepted,gate is neither defined in the circuit nor dec...


In [7]:
df = pd.read_parquet('./../results/part1/source_trace.parquet')
print(df.shape)
df.head(500)

(325625, 6)


,source_record_id,source_name,bronze_object,archive_member,record_locator,input_sha256
0,exp_dc1c8c81dc52,google_qec,bronze/source=google_qec/google-surface-code-c...,surface_code_bX_d3_r25_center_3_5,experiment_index=google_surface_code_bX_d3_r25...,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...
1,exp_b3206f3b7bf7,google_qec,bronze/source=google_qec/google-surface-code-c...,surface_code_bX_d3_r25_center_5_3,experiment_index=google_surface_code_bX_d3_r25...,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...
2,exp_2cee3fad4f1c,google_qec,bronze/source=google_qec/google-surface-code-c...,surface_code_bX_d3_r25_center_5_7,experiment_index=google_surface_code_bX_d3_r25...,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...
3,exp_a644e3a2d8d6,google_qec,bronze/source=google_qec/google-surface-code-c...,surface_code_bX_d3_r25_center_7_5,experiment_index=google_surface_code_bX_d3_r25...,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...
4,exp_fbfa44cc439a,google_qec,bronze/source=google_qec/google-surface-code-c...,surface_code_bX_d5_r25_center_5_5,experiment_index=google_surface_code_bX_d5_r25...,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...
...,...,...,...,...,...,...
495,google_shot_0b74acd34aec,google_qec,bronze/source=google_qec/google-surface-code-c...,surface_code_bX_d3_r25_center_3_5,shot_index=490,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...
496,google_shot_fced7b2bf972,google_qec,bronze/source=google_qec/google-surface-code-c...,surface_code_bX_d3_r25_center_3_5,shot_index=491,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...
497,google_shot_ec974a8d9eb0,google_qec,bronze/source=google_qec/google-surface-code-c...,surface_code_bX_d3_r25_center_3_5,shot_index=492,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...
498,google_shot_f2e9745c8513,google_qec,bronze/source=google_qec/google-surface-code-c...,surface_code_bX_d3_r25_center_3_5,shot_index=493,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...
